# MovieLens 100k Recommender System

This notebook builds a compact recommender system example using the MovieLens 100k dataset. The focus is clarity, sound methodology, and readable code rather than production complexity.

## Executive Summary

- **Problem:** Which movies should be shown first so each user is more likely to find something they enjoy?
- **Why it matters:** Better recommendations can improve discovery, engagement, and retention by helping users find relevant content faster.
- **Approach:** Compare a simple popularity baseline against a personalized collaborative filtering model built with `TruncatedSVD`, using a time-aware holdout for evaluation.
- **Key Result:** In this run, the SVD model achieved `Recall@10 = 0.1635` versus `0.0711` for the popularity baseline, a gain of `9.24 percentage points`.
- **Key Insight:** Personalization meaningfully improved top-of-list relevance in this offline test, while cold start and offline-to-online translation remain important limitations.

Dataset: [MovieLens 100k](https://grouplens.org/datasets/movielens/100k/)

## 1. Problem Framing

**Business question:** Which movies should be ranked highest for each user so relevant titles appear near the top of the list?

**Decision framing:** This is a ranking problem. The system is used to order movies so the user sees the most relevant options first, rather than scanning the full catalog.

**Why it matters:** Better ranking can improve discovery, engagement, and retention by reducing search friction and helping users find content faster.

**Success criteria:** In this notebook, success means placing a future liked movie inside the top recommendations, measured with `Recall@10`.

## Design Considerations

- **Relevance over raw popularity**: the target is personalized ranking quality, even though globally popular titles are often easier to recommend.
- **Simple baseline before complexity**: starting with popularity gives a clear benchmark, even if it is less personalized than collaborative filtering.
- **Collaborative filtering within a broader recommender landscape**: popularity-based methods are simple, content-based methods help with cold start, and collaborative filtering works well once interaction history exists; this notebook focuses on the collaborative filtering path.
- **Time-aware validation over random splitting**: this better matches real usage, even though it is slightly more restrictive than a random split.
- **SVD over heavier models**: matrix factorization is less expressive than deep learning approaches, but it is faster to build, easier to explain, and sufficient for a compact demo.
- **Single-metric clarity over evaluation breadth**: `Recall@10` is easy to interpret, even though it does not capture every aspect of recommendation quality.
- **Notebook clarity over production completeness**: the workflow is optimized for readability and discussion rather than full production realism.

## High-Level System Design

- **Offline**: train a collaborative filtering model such as SVD.
- **Store**: keep learned user and item embeddings for reuse at serving time.
- **Online**:
  - retrieve candidate items
  - rank them using the learned embeddings
  - return Top-K recommendations

In production, candidate generation is usually separated from ranking so retrieval can scale to much larger catalogs, often with ANN-style retrieval. Some systems also add a later re-ranking layer to balance relevance with diversity, freshness, or business rules.

In [73]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

from IPython.display import Markdown, display
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD

pd.set_option("display.max_colwidth", 60)
np.random.seed(42)

## 2. Data Loading & Overview

The notebook will automatically download the official MovieLens 100k dataset into `data/` if the files are missing.

- Dataset page: https://grouplens.org/datasets/movielens/100k/
- Direct download: https://files.grouplens.org/datasets/movielens/ml-100k.zip

After download and extraction, the notebook uses these files:

- `data/u.data`: the core interaction table with user IDs, movie IDs, ratings, and timestamps
- `data/u.item`: movie metadata, used here mainly to map movie IDs to titles
- `data/u.user`: user metadata, used here only for a basic dataset overview

The MovieLens 100k package also includes other files that are not needed here, for example:

- `u.genre`: maps genre names to genre IDs
- `u.occupation`: lists occupation categories used in `u.user`
- `u.info`: summary counts for users, items, and ratings
- `u1.base` / `u1.test` and similar files: pre-built train/test splits provided with the dataset

Those files are useful for richer feature engineering or benchmark experiments, but this notebook keeps the scope narrow and builds its own simple time-based split from the main ratings table.

In [74]:
MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

required_files = ["u.data", "u.item", "u.user"]

if not all((data_dir / file_name).exists() for file_name in required_files):
    zip_path = data_dir / "ml-100k.zip"
    extract_dir = data_dir / "ml-100k"

    print(f"Downloading MovieLens 100k from {MOVIELENS_URL} ...")
    urlretrieve(MOVIELENS_URL, zip_path)

    with ZipFile(zip_path, "r") as zip_file:
        zip_file.extractall(data_dir)

    for file_name in required_files:
        source = extract_dir / file_name
        destination = data_dir / file_name
        if source.exists() and not destination.exists():
            destination.write_bytes(source.read_bytes())

if not all((data_dir / file_name).exists() for file_name in required_files):
    missing_files = [file_name for file_name in required_files if not (data_dir / file_name).exists()]
    raise FileNotFoundError(
        "MovieLens 100k files are still missing after download attempt: "
        f"{missing_files}. Expected them under {data_dir.resolve()}"
    )

ratings = pd.read_csv(
    data_dir / "u.data",
    sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"],
)

movies = pd.read_csv(
    data_dir / "u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["item_id", "title"],
)

users = pd.read_csv(
    data_dir / "u.user",
    sep="|",
    header=None,
    names=["user_id", "age", "gender", "occupation", "zip_code"],
)

ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")
ratings = ratings.merge(movies[["item_id", "title"]], on="item_id", how="left")

print(f"Using data from: {data_dir.resolve()}")
print(f"Ratings shape: {ratings.shape}")
print(f"Users: {users['user_id'].nunique()}")
print(f"Movies: {movies['item_id'].nunique()}")

Using data from: /home/phrey/projects/ml-casebook/ml-examples/recommender/data
Ratings shape: (100000, 5)
Users: 943
Movies: 1682


In [75]:
overview = pd.DataFrame(
    {
        "metric": ["ratings", "users", "movies", "avg_rating"],
        "value": [
            len(ratings),
            ratings["user_id"].nunique(),
            ratings["item_id"].nunique(),
            round(ratings["rating"].mean(), 2),
        ],
    }
)

overview

,metric,value
0,ratings,100000.00
1,users,943.00
2,movies,1682.00
3,avg_rating,3.53


In [76]:
interaction_density = len(ratings) / (ratings["user_id"].nunique() * ratings["item_id"].nunique())
avg_ratings_per_user = ratings.groupby("user_id")["item_id"].size().mean()

display(Markdown(
    f"Interpretation: this is a moderately sized but sparse recommendation dataset, with about **{interaction_density:.2%}** of the user-item matrix observed. "
    f"On average, each user has rated **{avg_ratings_per_user:.1f}** movies, which is enough to demonstrate collaborative filtering."
))


Interpretation: this is a moderately sized but sparse recommendation dataset, with about **6.30%** of the user-item matrix observed. On average, each user has rated **106.0** movies, which is enough to demonstrate collaborative filtering.

## 3. Data Preprocessing

We keep preprocessing intentionally simple:

- map raw user and movie IDs to zero-based matrix indices
- create the training interaction matrix after the split
- keep movie titles in a lookup table for readable outputs

In [77]:
user_ids = np.sort(ratings["user_id"].unique())
item_ids = np.sort(ratings["item_id"].unique())

user_to_index = {user_id: idx for idx, user_id in enumerate(user_ids)}
item_to_index = {item_id: idx for idx, item_id in enumerate(item_ids)}

ratings["user_idx"] = ratings["user_id"].map(user_to_index)
ratings["item_idx"] = ratings["item_id"].map(item_to_index)

movie_lookup = movies.set_index("item_id")

## 4. Validation Strategy

For demo simplicity, this notebook uses a single train-test split rather than a separate train-validation-test workflow. Recommendation systems should still be evaluated on future behavior rather than randomly shuffled behavior.

To keep the logic simple and reasonably realistic, we use a time-aware train-test split: for each eligible user, we hold out their latest positively rated movie (`rating >= 4`) as the test item and train only on interactions that happened earlier.


In [78]:
POSITIVE_RATING = 4
MIN_USER_INTERACTIONS = 5


def make_time_based_split(frame, positive_rating=4, min_user_interactions=5):
    frame = frame.sort_values(["user_id", "timestamp", "item_id"]).copy()
    train_parts = []
    test_rows = []

    for _, user_history in frame.groupby("user_id", sort=False):
        if len(user_history) < min_user_interactions:
            train_parts.append(user_history)
            continue

        positive_history = user_history[user_history["rating"] >= positive_rating]
        if positive_history.empty:
            train_parts.append(user_history)
            continue

        test_row = positive_history.iloc[-1]
        test_position = user_history.index.get_loc(test_row.name)
        if test_position == 0:
            train_parts.append(user_history)
            continue

        # Keep only interactions that happened before the held-out event to avoid temporal leakage.
        train_parts.append(user_history.iloc[:test_position])
        test_rows.append(test_row)

    train = pd.concat(train_parts, ignore_index=True)
    test = pd.DataFrame(test_rows).reset_index(drop=True)
    return train, test


train_ratings, test_ratings = make_time_based_split(
    ratings,
    positive_rating=POSITIVE_RATING,
    min_user_interactions=MIN_USER_INTERACTIONS,
)

print(f"Train ratings: {len(train_ratings):,}")
print(f"Test ratings: {len(test_ratings):,}")
print(f"Users with a held-out positive item: {test_ratings["user_id"].nunique():,}")


Train ratings: 96,929
Test ratings: 942
Users with a held-out positive item: 942


In [79]:
test_share = len(test_ratings) / len(ratings)
coverage = test_ratings["user_id"].nunique() / ratings["user_id"].nunique()

display(Markdown(
    f"Interpretation: about **{1 - test_share:.1%}** of interactions remain in training, while one future positive item is held out for about **{coverage:.1%}** of users. "
    "This avoids leaking later interactions into training, but it also leaves less data for fitting the model."
))


Interpretation: about **99.1%** of interactions remain in training, while one future positive item is held out for about **99.9%** of users. This avoids leaking later interactions into training, but it also leaves less data for fitting the model.

## 5. Baseline: Popularity-Based Recommendations

A good baseline answers a simple question: can we beat a naive strategy? Here the baseline recommends movies that received the most positive feedback in the training set. This ignores personalization, but it is easy to explain and often surprisingly competitive.

In [80]:
def build_popularity_table(frame, positive_rating=4):
    positive = frame[frame["rating"] >= positive_rating]
    popularity = (
        positive.groupby("item_id")
        .agg(
            positive_ratings=("rating", "size"),
            avg_rating=("rating", "mean"),
        )
        .sort_values(["positive_ratings", "avg_rating"], ascending=False)
    )
    return popularity


popularity_table = build_popularity_table(train_ratings, positive_rating=POSITIVE_RATING)


def recommend_popular(user_id, train_frame, popularity_frame, k=10):
    seen_items = set(train_frame.loc[train_frame["user_id"] == user_id, "item_id"])
    ranked_items = [item for item in popularity_frame.index if item not in seen_items]
    return ranked_items[:k]


popularity_table.head(10).merge(movie_lookup[["title"]], left_index=True, right_index=True)

,positive_ratings,avg_rating,title
item_id,,,
50,498,4.648594,Star Wars (1977)
100,402,4.559701,Fargo (1996)
181,373,4.453083,Return of the Jedi (1983)
127,349,4.613181,"Godfather, The (1972)"
174,347,4.579251,Raiders of the Lost Ark (1981)
98,341,4.524927,"Silence of the Lambs, The (1991)"
258,335,4.388060,Contact (1997)
1,315,4.374603,Toy Story (1995)
286,295,4.444068,"English Patient, The (1996)"


In [81]:
top_popular_titles = ", ".join(
    popularity_table.head(3).merge(movie_lookup[["title"]], left_index=True, right_index=True)["title"].tolist()
)

display(Markdown(
    f"Interpretation: the popularity baseline recommends broadly popular, well-liked titles such as **{top_popular_titles}**. It is simple and often hard to beat, but it does not adapt to individual user taste."
))


Interpretation: the popularity baseline recommends broadly popular, well-liked titles such as **Star Wars (1977), Fargo (1996), Return of the Jedi (1983)**. It is simple and often hard to beat, but it does not adapt to individual user taste.

## 6. Personalized Model: Collaborative Filtering via SVD

We train a simple matrix factorization model with `TruncatedSVD`. SVD is used to learn latent user and item factors, capturing hidden preferences in sparse interaction data. It provides a strong, interpretable baseline before moving to more complex deep learning models.

The idea is to compress the sparse user-item matrix into a smaller latent space:

- each user gets an embedding that summarizes preference patterns
- each movie gets an embedding that summarizes who tends to like it
- recommendations come from matching user and movie embeddings

In [82]:
def build_train_interaction_matrix(frame, user_ids, item_ids):
    matrix = (
        frame.pivot_table(
            index="user_id",
            columns="item_id",
            values="rating",
            fill_value=0,
        )
        .reindex(index=user_ids, columns=item_ids, fill_value=0)
        .to_numpy(dtype=float)
    )
    return matrix


train_interaction_matrix = build_train_interaction_matrix(train_ratings, user_ids, item_ids)

# Keep the latent dimension well below the matrix width for a compact demo model.
n_components = 30
svd = TruncatedSVD(n_components=n_components, random_state=42)
user_embeddings = svd.fit_transform(train_interaction_matrix)
item_embeddings = svd.components_.T
score_matrix = user_embeddings @ item_embeddings.T

print(f"Train interaction matrix shape: {train_interaction_matrix.shape}")
print(f"Latent factors: {n_components}")

Train interaction matrix shape: (943, 1682)
Latent factors: 30


In [83]:
display(Markdown(
    "Interpretation: the training matrix is still small enough for a compact notebook demo, while **30 latent factors** gives the model enough room to learn broad preference patterns without making the setup unnecessarily complex. "
    "Recommendations are scored with a **dot product** between user and item embeddings. Dot product keeps magnitude, so it can reflect how strongly a user prefers an item, while cosine similarity would keep only directional similarity and lose that preference-strength signal."
))


Interpretation: the training matrix is still small enough for a compact notebook demo, while **30 latent factors** gives the model enough room to learn broad preference patterns without making the setup unnecessarily complex. Recommendations are scored with a **dot product** between user and item embeddings. Dot product keeps magnitude, so it can reflect how strongly a user prefers an item, while cosine similarity would keep only directional similarity and lose that preference-strength signal.

In [84]:
def recommend_svd(user_id, train_frame, score_matrix, user_to_index, item_ids, k=10):
    user_idx = user_to_index[user_id]
    scores = score_matrix[user_idx].copy()

    seen_items = train_frame.loc[train_frame["user_id"] == user_id, "item_id"].values
    seen_indices = [item_to_index[item_id] for item_id in seen_items]
    scores[seen_indices] = -np.inf

    top_indices = np.argsort(scores)[::-1][:k]
    return [item_ids[idx] for idx in top_indices]


sample_user_id = int(test_ratings.iloc[0]["user_id"])
sample_recs = recommend_svd(sample_user_id, train_ratings, score_matrix, user_to_index, item_ids, k=10)
pd.DataFrame({"item_id": sample_recs, "title": movie_lookup.loc[sample_recs, "title"].values})

,item_id,title
0,475,Trainspotting (1996)
1,474,Dr. Strangelove or: How I Learned to Stop Worrying and L...
2,423,E.T. the Extra-Terrestrial (1982)
3,276,Leaving Las Vegas (1995)
4,732,Dave (1993)
5,483,Casablanca (1942)
6,655,Stand by Me (1986)
7,403,Batman (1989)
8,433,Heathers (1989)
9,273,Heat (1995)


In [85]:
display(Markdown(
    "Interpretation: these recommendations are personalized because they are generated from the user's latent preference vector and exclude movies already seen in training."
))


Interpretation: these recommendations are personalized because they are generated from the user's latent preference vector and exclude movies already seen in training.

## 7. Performance Summary

`Recall@10` measures whether the true held-out item appears in the top-10 recommendations. A higher value means the recommender is better at surfacing relevant items near the top of the list.

Formula:

`Recall@K = (number of users where the held-out item appears in the top K) / (total number of evaluated users)`

Because we hold out one positive movie per user, `Recall@10` here is equivalent to hit rate at 10.

In [86]:
def recall_at_k(test_frame, recommender_fn, k=10):
    hits = []
    for row in test_frame.itertuples(index=False):
        recommended_items = recommender_fn(row.user_id, k)
        hits.append(int(row.item_id in recommended_items))
    return float(np.mean(hits))


popularity_recall = recall_at_k(
    test_ratings,
    recommender_fn=lambda user_id, k: recommend_popular(user_id, train_ratings, popularity_table, k=k),
    k=10,
)

svd_recall = recall_at_k(
    test_ratings,
    recommender_fn=lambda user_id, k: recommend_svd(user_id, train_ratings, score_matrix, user_to_index, item_ids, k=k),
    k=10,
)

pd.DataFrame(
    {
        "model": ["Popularity baseline", "SVD collaborative filtering"],
        "Recall@10": [round(popularity_recall, 4), round(svd_recall, 4)],
    }
).sort_values("Recall@10", ascending=False)

,model,Recall@10
1,SVD collaborative filtering,0.1614
0,Popularity baseline,0.0701


In [87]:
if svd_recall > popularity_recall:
    display(Markdown(
        f"**Result:** the personalized SVD model outperformed the popularity baseline by **{(svd_recall - popularity_recall) * 100:.2f} percentage points** in Recall@10 (**{(svd_recall / popularity_recall - 1):+.1%} relative lift**)."
    ))
elif svd_recall < popularity_recall:
    display(Markdown(
        f"**Result:** the popularity baseline outperformed the personalized SVD model by **{(popularity_recall - svd_recall) * 100:.2f} percentage points** in Recall@10 (**{(popularity_recall / svd_recall - 1):+.1%} relative lift**)."
    ))
else:
    display(Markdown("**Result:** both models achieved the same Recall@10."))


**Result:** the personalized SVD model outperformed the popularity baseline by **9.13 percentage points** in Recall@10 (**+130.3% relative lift**).

## 8. Business Interpretation

This summary translates the evaluation result into plain business language. The key question is whether personalization materially improves the chance of showing a user a movie they will actually like.

Below is a compact plain-language summary of the benchmark result.


The summary below keeps the result readable without using a wide table.


In [88]:
best_model = "SVD collaborative filtering" if svd_recall >= popularity_recall else "Popularity baseline"
absolute_lift = svd_recall - popularity_recall
relative_lift = (svd_recall / popularity_recall - 1) if popularity_recall > 0 else np.nan

display(Markdown(f"**Best-performing approach:** {best_model}"))
display(Markdown("**Recall@10 here means:** how often the held-out movie appeared in the top 10 recommendations."))
display(Markdown(
    f"**Business interpretation:** the personalized SVD model improved Recall@10 by **{absolute_lift * 100:.2f} percentage points** versus the popularity baseline "
    f"(**{relative_lift:+.1%} relative lift**). Positive lift means the system is better at surfacing relevant movies near the top of the list."
))


**Best-performing approach:** SVD collaborative filtering

**Recall@10 here means:** how often the held-out movie appeared in the top 10 recommendations.

**Business interpretation:** the personalized SVD model improved Recall@10 by **9.13 percentage points** versus the popularity baseline (**+130.3% relative lift**). Positive lift means the system is better at surfacing relevant movies near the top of the list.

## 9. Recommendation Examples

Below we inspect a couple of users to make the output concrete. For each user, we show some previously watched movies from the training period and the model's top recommendations.


In [89]:
def show_user_examples(user_id, train_frame, test_frame, k=10, history_n=8):
    history = (
        train_frame.loc[train_frame["user_id"] == user_id, ["title", "rating", "timestamp"]]
        .sort_values("timestamp", ascending=False)
        .head(history_n)
        .reset_index(drop=True)
    )

    rec_item_ids = recommend_svd(user_id, train_frame, score_matrix, user_to_index, item_ids, k=k)
    recommendations = pd.DataFrame(
        {
            "item_id": rec_item_ids,
            "recommended_title": movie_lookup.loc[rec_item_ids, "title"].values,
        }
    )

    held_out = test_frame.loc[test_frame["user_id"] == user_id, ["title", "rating"]].reset_index(drop=True)
    return history, held_out, recommendations


example_users = test_ratings["user_id"].drop_duplicates().sort_values().head(2).tolist()

for user_id in example_users:
    print(f"\nUser {user_id}")
    history, held_out, recommendations = show_user_examples(user_id, train_ratings, test_ratings, k=10)
    print("Previously watched (most recent first):")
    display(history)
    print("Held-out future positive item:")
    display(held_out)
    print("Top recommendations:")
    display(recommendations)


User 1
Previously watched (most recent first):


,title,rating,timestamp
0,Copycat (1995),3,1998-03-13 01:15:12
1,Delicatessen (1991),5,1998-03-13 01:15:11
2,"Truth About Cats & Dogs, The (1996)",5,1998-03-13 01:15:11
3,Kolya (1996),5,1998-03-13 01:13:53
4,"Grand Day Out, A (1992)",3,1998-03-01 06:15:28
5,Crumb (1994),5,1998-03-01 06:15:09
6,This Is Spinal Tap (1984),4,1998-03-01 06:15:08
7,Gattaca (1997),5,1998-03-01 06:13:47


Held-out future positive item:


,title,rating
0,When the Cats Away (Chacun cherche son chat) (1996),4


Top recommendations:


...

,title,rating,timestamp
0,Evita (1996),3,1998-03-04 02:42:33
1,Fly Away Home (1996),4,1998-03-04 02:39:57
2,Air Force One (1997),4,1998-03-04 02:39:57
3,"Rainmaker, The (1997)",4,1998-03-04 02:37:41
4,Good Will Hunting (1997),5,1998-03-04 02:37:41
5,Emma (1996),5,1998-02-27 04:01:24
6,Star Wars (1977),5,1998-02-27 04:01:24
7,"Godfather, The (1972)",5,1998-02-27 04:01:24


Held-out future positive item:


,title,rating
0,As Good As It Gets (1997),5


Top recommendations:


,item_id,recommended_title
0,515,"Boot, Das (1981)"
1,124,Lone Star (1996)
2,15,Mr. Holland's Opus (1995)
3,181,Return of the Jedi (1983)
4,137,Big Night (1996)
5,9,Dead Man Walking (1995)
6,340,Boogie Nights (1997)
7,690,Seven Years in Tibet (1997)
8,116,Cold Comfort Farm (1995)
9,750,Amistad (1997)


In [90]:
example_hits = []
for user_id in example_users:
    held_out_item = test_ratings.loc[test_ratings["user_id"] == user_id, "item_id"].iloc[0]
    top_recs = recommend_svd(user_id, train_ratings, score_matrix, user_to_index, item_ids, k=10)
    example_hits.append(held_out_item in top_recs)

display(Markdown(
    "Interpretation: these user examples are deterministic snapshots chosen for readability rather than for best-case performance. "
    "They are meant to make the recommendation logic concrete, while the overall model judgment should come from the full-test Recall@10 benchmark above."
))


Interpretation: these user examples are deterministic snapshots chosen for readability rather than for best-case performance. They are meant to make the recommendation logic concrete, while the overall model judgment should come from the full-test Recall@10 benchmark above.

## 10. Key Takeaways

- A simple popularity baseline is essential because it gives a clear benchmark before claiming personalization value.
- In this notebook, collaborative filtering improved top-of-list relevance versus the baseline, suggesting that user-level preference patterns are informative.
- The time-aware validation strategy makes the offline result more credible than a random split would.
- The main gaps to solve next are cold start, richer feedback signals, and validating impact in an online setting.

## Limitations and Risks

This notebook is intentionally scoped as a compact demo rather than a full recommendation platform.

- **Sparsity**: user-item data is sparse, so many preferences are only weakly observed.
- **Popularity bias**: the model can favor movies that already receive more attention.
- **Limited feedback type**: this notebook uses explicit ratings only and ignores richer implicit signals such as clicks, views, and watch time.
- **Simplified evaluation**: it holds out one positive item per user, which is simple to explain but not exhaustive.
- **Cold start**: it does not address brand new users or brand new movies well.
- **No side features**: it does not use user demographics, item metadata, or contextual features that could improve recommendations.
- **Offline metric limitation**: `Recall@10` is useful for benchmarking, but real-world impact still needs online testing.
- **Narrow optimization target**: it focuses on relevance only, not diversity, novelty, fairness, or business constraints.


## Production Considerations

If this moved beyond a notebook, common next steps would be:

| Architecture | What it does | When it fits |
|---|---|---|
| Current notebook: single-stage scoring | Score items directly for each user and rank the results in one step. | Small catalogs, demos, or simple systems. |
| Two-stage: candidate generation + ranking | First narrow the catalog to likely items, then rank that smaller set more precisely. | Larger catalogs where scoring every item is too expensive. |
| Three-stage: candidate generation + ranking + re-ranking | Add a final step to adjust the ranked list for diversity, freshness, fairness, or business rules. | More mature systems with multiple objectives beyond relevance alone. |

Other common next steps include:

- combine collaborative filtering with content or metadata features to reduce cold-start problems
- use retrieval infrastructure such as ANN indexes for fast candidate generation on large catalogs
- incorporate implicit feedback and recency signals instead of relying only on explicit ratings
- consider stronger models such as Alternating Least Squares (ALS) for matrix factorization on implicit feedback, Bayesian Personalized Ranking (BPR) for pairwise ranking, two-tower models for retrieval, or richer ranking models
- evaluate online with A/B tests and monitor recommendation quality over time


## References

- MovieLens 100k dataset: https://grouplens.org/datasets/movielens/100k/
- Collaborative filtering overview: https://developers.google.com/machine-learning/recommendation/collaborative/basics
- `TruncatedSVD` documentation: https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html
- Precision and recall overview: https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html